# Kepler Exoplanet Classification — complete code notebook
**CMPS 261 (Machine Learning), Track A — NASA Kepler Objects of Interest** · Joe Khoury, Andy Sleiman, Anthony Merhy

This notebook contains **all the code** of the final project: data loading and cleaning, leakage audit, grouped splits, feature engineering, hyper-parameter tuning, model selection, failure analysis, mitigations, SHAP, complexity analysis, the single held-out test evaluation, the CANDIDATE ranking and the sensitivity analysis. Run all cells top to bottom (about 45 min on 2 CPU cores). Repository: https://github.com/AndySleiman/kepler-exoplanet-classification

| Part | What it does | Report section |
|---|---|---|
| 0 | setup, shared module (data, split, features, metrics), PyTorch model | 3 |
| 1 | tune LR / random forest / XGBoost / sklearn-MLP, with and without the 5 new features | 3, 4 |
| 2 | tune the PyTorch residual MLP | 4 |
| 3 | model selection, failure analysis, fixes, SHAP, complexity (**validation only**) | 5 |
| 3b | dataset statistics and leakage ablation | 3 |
| 4 | freeze → **one** test evaluation → rank CANDIDATE objects | 5 |
| 5 | report figures | 5 |
| 6 | extra mitigations (deferral, specialist) and interpretability trade-off (validation only) | 5 |
| 7 | sensitivity to the split and evaluation design (validation only) | 5 |

The saved outputs are from a fresh-kernel rerun with the pinned Python 3.12 environment. The original report results and trained models are kept separately from changed rerun files in `results/reproduction/`. The compact-feature and calibration-correction comparisons are exploratory for the reasons described in the report.


In [1]:
import os, sys
# the shared modules are written to ../final/ by the next two cells and imported from there
FINAL = os.path.abspath("../final"); os.makedirs(FINAL, exist_ok=True); os.makedirs("cache", exist_ok=True)
sys.path.insert(0, FINAL); os.environ["PYTHONPATH"] = FINAL + os.pathsep + os.environ.get("PYTHONPATH", "")   # also for joblib worker processes

## 0. Shared module: data, grouped split, preprocessing, engineered features, metrics
Written to `final/common.py` so that saved scikit-learn pipelines (which reference `common.log_and_engineer`) can be loaded by `predict.py`.

In [2]:
%%writefile ../final/common.py
"""Shared setup for the final experiments: data, split, folds, metrics, feature engineering.
Uses exactly the same split (seed 42, grouped by star) as the progress-report notebook."""
import time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score,
                             brier_score_loss, precision_recall_curve)
warnings.filterwarnings("ignore")
RS = 42
DATA_PATH = "../data/kepler_cumulative.csv"

FEATS = ["koi_period", "koi_duration", "koi_depth", "koi_impact", "koi_model_snr",
         "koi_prad", "koi_teq", "koi_insol", "koi_steff", "koi_slogg", "koi_srad", "koi_kepmag"]
LOG_COLS = ["koi_period", "koi_duration", "koi_depth", "koi_model_snr",
            "koi_prad", "koi_insol", "koi_srad", "koi_teq"]

# ---- new physically motivated features (computed from the 12 base columns only) ----
NEW_FEATS = ["depth_consistency", "duration_ratio", "implied_noise", "star_density", "rp_over_rs"]
ALL_FEATS = FEATS + NEW_FEATS
R_SUN_IN_R_EARTH = 109.1
KEPLER_BASELINE_DAYS = 1460.0

def add_features(X):
    """depth_consistency : observed depth / depth expected from (Rp/Rs)^2. ~1 for a real planet-sized dip,
                           far from 1 when the 'planet radius' was derived from a contaminated/ eclipsing signal.
       duration_ratio    : observed duration / duration expected for a circular central transit around a star of
                           this density (T ~ P^(1/3) * rho^(-1/3)).  Eclipsing binaries on other stars deviate.
       implied_noise     : per-point noise level implied by depth, SNR and the number of transits (weak-signal indicator).
       star_density      : relative stellar density from log g and radius (rho ~ g / R).
       rp_over_rs        : planet-to-star radius ratio."""
    X = X.copy()
    rp_rs = (X["koi_prad"] / (X["koi_srad"] * R_SUN_IN_R_EARTH)).clip(lower=1e-6)
    X["rp_over_rs"] = np.log10(rp_rs)
    expected_depth_ppm = (rp_rs ** 2) * 1e6
    X["depth_consistency"] = np.log10((X["koi_depth"].clip(lower=1e-3) / expected_depth_ppm).clip(lower=1e-3))
    rho = (10 ** X["koi_slogg"]) / X["koi_srad"].clip(lower=1e-3)          # ~ density up to a constant
    X["star_density"] = np.log10(rho.clip(lower=1e-3))
    expected_dur = (X["koi_period"].clip(lower=1e-3) ** (1 / 3)) * (rho.clip(lower=1e-3) ** (-1 / 3))
    X["duration_ratio"] = np.log10((X["koi_duration"].clip(lower=1e-3) / expected_dur).clip(lower=1e-3))
    n_tr = (KEPLER_BASELINE_DAYS / X["koi_period"].clip(lower=1e-3)).clip(lower=1)
    X["implied_noise"] = np.log10((X["koi_depth"].clip(lower=1e-3) * np.sqrt(n_tr) / X["koi_model_snr"].clip(lower=1e-3)).clip(lower=1e-3))
    return X

def log_transform(X):
    X = X.copy()
    for c in LOG_COLS:
        if c in X:
            X[c] = np.log10(X[c].clip(lower=1e-3))
    return X

def log_and_engineer(X):
    return log_transform(add_features(X))     # engineered features use RAW units, then logs are applied

def preprocessing(scale, engineered=False):
    steps = [("log", FunctionTransformer(log_and_engineer if engineered else log_transform)),
             ("imp", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("sc", StandardScaler()))
    return steps

def recall_at_precision(y_true, p, target=0.95):
    prec, rec, _ = precision_recall_curve(y_true, p)
    ok = rec[prec >= target]
    return float(ok.max()) if len(ok) else 0.0

def all_metrics(y_true, p, thr=0.5):
    return {"PR-AUC": average_precision_score(y_true, p), "ROC-AUC": roc_auc_score(y_true, p),
            "F1": f1_score(y_true, p >= thr), "Recall@P95": recall_at_precision(y_true, p),
            "Brier": brier_score_loss(y_true, p)}

def load():
    df = pd.read_csv(DATA_PATH, comment="#")
    lab = df[df["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE"])].copy()
    cand = df[df["koi_disposition"] == "CANDIDATE"].copy()
    lab["y"] = (lab["koi_disposition"] == "CONFIRMED").astype(int)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RS)
    tr_idx, te_idx = next(gss.split(lab, lab["y"], lab["kepid"]))
    tr, te = lab.iloc[tr_idx].copy(), lab.iloc[te_idx].copy()
    y = tr["y"].values
    splits = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RS).split(tr, y, tr["kepid"]))
    return df, lab, cand, tr, te, y, splits


Overwriting ../final/common.py


### PyTorch residual MLP (scikit-learn-compatible wrapper)

In [3]:
%%writefile ../final/torch_model.py
"""PyTorch residual MLP for tabular data, wrapped as a scikit-learn estimator.
Differences from the sklearn MLP of the progress report: residual blocks + LayerNorm, SiLU, dropout,
AdamW with weight decay, one-cycle learning-rate schedule, a fixed epoch budget (no early stopping on
validation data) and an ensemble of several random seeds."""
import numpy as np, torch, torch.nn as nn
from sklearn.base import BaseEstimator, ClassifierMixin

torch.set_num_threads(2)

class Block(nn.Module):
    def __init__(self, d, p):
        super().__init__()
        self.f = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.SiLU(), nn.Dropout(p), nn.Linear(d, d))
    def forward(self, x):
        return x + self.f(x)

class Net(nn.Module):
    def __init__(self, d_in, width, n_blocks, p):
        super().__init__()
        self.inp = nn.Linear(d_in, width)
        self.blocks = nn.Sequential(*[Block(width, p) for _ in range(n_blocks)])
        self.out = nn.Sequential(nn.LayerNorm(width), nn.SiLU(), nn.Linear(width, 1))
    def forward(self, x):
        return self.out(self.blocks(self.inp(x))).squeeze(-1)

class TorchMLP(BaseEstimator, ClassifierMixin):
    def __init__(self, width=128, n_blocks=2, dropout=0.1, lr=3e-3, weight_decay=1e-3,
                 epochs=60, batch_size=128, n_seeds=3, random_state=42):
        self.width, self.n_blocks, self.dropout, self.lr = width, n_blocks, dropout, lr
        self.weight_decay, self.epochs, self.batch_size = weight_decay, epochs, batch_size
        self.n_seeds, self.random_state = n_seeds, random_state

    def fit(self, X, y, sample_weight=None):
        X = torch.tensor(np.asarray(X), dtype=torch.float32)
        yt = torch.tensor(np.asarray(y), dtype=torch.float32)
        w = torch.ones(len(yt)) if sample_weight is None else torch.tensor(np.asarray(sample_weight), dtype=torch.float32)
        self.classes_ = np.array([0, 1])
        self.nets_ = []
        n = len(yt); steps = int(np.ceil(n / self.batch_size)) * self.epochs
        for s in range(self.n_seeds):
            g = torch.Generator().manual_seed(self.random_state + s)
            torch.manual_seed(self.random_state + s)
            net = Net(X.shape[1], self.width, self.n_blocks, self.dropout)
            opt = torch.optim.AdamW(net.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=self.lr, total_steps=steps)
            net.train()
            for _ in range(self.epochs):
                perm = torch.randperm(n, generator=g)
                for i in range(0, n, self.batch_size):
                    idx = perm[i:i + self.batch_size]
                    if len(idx) < 2: continue
                    loss = (nn.functional.binary_cross_entropy_with_logits(net(X[idx]), yt[idx], reduction="none") * w[idx]).mean()
                    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            net.eval(); self.nets_.append(net)
        return self

    def predict_proba(self, X):
        X = torch.tensor(np.asarray(X), dtype=torch.float32)
        with torch.no_grad():
            p = np.mean([torch.sigmoid(n(X)).numpy() for n in self.nets_], axis=0)
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

    def n_parameters(self):
        return sum(p.numel() for p in self.nets_[0].parameters()) * len(self.nets_)


Overwriting ../final/torch_model.py


## 1. Tune the classical models and the sklearn MLP (base 12 features vs 12 + 5 new features)
Level-1 baselines (logistic regression, random forest, XGBoost) and a first neural baseline, all tuned for PR-AUC with the same 5 grouped folds. **Validation only.**

In [4]:
"""Step 1: re-run the model ladder on the current data (base 12 features vs 12 + 5 engineered features).
All tuning uses ONLY the 80% train/validation part (5-fold grouped CV). The test set is not touched."""
import sys, pickle, time
sys.path.insert(0, ".")
from common import *
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

df, lab, cand, tr, te, y, splits = load()
BASE12 = FEATS

def run(name, pipe, space, n_iter, engineered):
    X = tr[BASE12]
    if n_iter is None:
        s = GridSearchCV(pipe, space, scoring="average_precision", cv=splits, n_jobs=2)
    else:
        s = RandomizedSearchCV(pipe, space, n_iter=n_iter, scoring="average_precision", cv=splits, n_jobs=2, random_state=RS)
    t0 = time.time(); s.fit(X, y); tune = time.time() - t0
    best = s.best_estimator_
    p = cross_val_predict(best, X, y, cv=splits, method="predict_proba")[:, 1]
    t0 = time.time(); best.fit(X, y); fit = time.time() - t0
    m = all_metrics(y, p)
    m["PR-AUC fold SD"] = float(np.std([average_precision_score(y[v], p[v]) for _, v in splits]))
    m.update({"Configs tried": len(s.cv_results_["params"]), "Tuning time (s)": tune, "Fit time (s)": fit})
    print(f"{name:34s} PR-AUC {m['PR-AUC']:.4f}  R@P95 {m['Recall@P95']:.3f}  Brier {m['Brier']:.4f}  tune {tune:.0f}s", flush=True)
    return dict(name=name, metrics=m, oof=p, model=best, params=s.best_params_)

LR = lambda: LogisticRegression(max_iter=5000, class_weight="balanced")
RF = lambda: RandomForestClassifier(n_jobs=1, random_state=RS)
XG = lambda: XGBClassifier(eval_metric="logloss", n_jobs=1, random_state=RS, tree_method="hist")
ML = lambda: MLPClassifier(max_iter=600, early_stopping=True, validation_fraction=0.15, n_iter_no_change=25, random_state=RS)
spaces = {
 "Logistic regression": (LR, True, {"m__C": [0.001, 0.01, 0.1, 1, 10, 100], "m__penalty": ["l2"]}, None),
 "Random forest": (RF, False, {"m__n_estimators": [300, 600], "m__max_depth": [None, 10, 20], "m__min_samples_leaf": [1, 3, 5, 10], "m__max_features": ["sqrt", 0.5]}, 12),
 "XGBoost": (XG, False, {"m__n_estimators": [200, 400, 800], "m__max_depth": [3, 4, 6, 8], "m__learning_rate": [0.02, 0.05, 0.1], "m__subsample": [0.7, 0.85, 1.0], "m__colsample_bytree": [0.6, 0.8, 1.0], "m__min_child_weight": [1, 3, 5], "m__reg_lambda": [0.5, 1, 5]}, 40),
 "MLP (sklearn)": (ML, True, {"m__hidden_layer_sizes": [(64,), (128, 64), (256, 128, 64), (64, 64)], "m__alpha": [1e-4, 1e-3, 1e-2, 1e-1], "m__learning_rate_init": [1e-3, 3e-3], "m__batch_size": [64, 256]}, 20),
}
out = {}
for engineered in (False, True):
    for name, (mk, scale, space, n_iter) in spaces.items():
        tag = name + (" +new features" if engineered else "")
        out[tag] = run(tag, Pipeline(preprocessing(scale, engineered) + [("m", mk())]), space, n_iter, engineered)
        pickle.dump(out, open("cache/s01_classical.pkl", "wb"))
print("DONE")


Logistic regression                PR-AUC 0.7258  R@P95 0.000  Brier 0.1592  tune 2s


Random forest                      PR-AUC 0.9433  R@P95 0.651  Brier 0.0628  tune 105s


XGBoost                            PR-AUC 0.9550  R@P95 0.741  Brier 0.0541  tune 27s


MLP (sklearn)                      PR-AUC 0.9516  R@P95 0.693  Brier 0.0588  tune 93s


Logistic regression +new features  PR-AUC 0.7374  R@P95 0.004  Brier 0.1562  tune 1s


Random forest +new features        PR-AUC 0.9545  R@P95 0.722  Brier 0.0562  tune 133s


XGBoost +new features              PR-AUC 0.9607  R@P95 0.769  Brier 0.0516  tune 35s


MLP (sklearn) +new features        PR-AUC 0.9544  R@P95 0.721  Brier 0.0562  tune 91s


DONE


## 2. Tune the PyTorch residual MLP (Level 2)
Random search over width, depth, dropout, learning rate, weight decay and epochs; final model = 5-seed ensemble. **Validation only.**

In [5]:
"""Step 2: tune the PyTorch residual MLP with the same grouped 5-fold CV (random search, 10 configs)."""
import sys, pickle, time, itertools
sys.path.insert(0, ".")
from common import *
from torch_model import TorchMLP
from sklearn.model_selection import cross_val_predict, ParameterSampler

df, lab, cand, tr, te, y, splits = load()
space = {"m__width": [64, 128, 256], "m__n_blocks": [1, 2, 3], "m__dropout": [0.0, 0.1, 0.2, 0.3],
         "m__lr": [1e-3, 3e-3, 1e-2], "m__weight_decay": [1e-4, 1e-3, 1e-2, 1e-1], "m__epochs": [40, 80]}
out = {}
for engineered in (True,):
    tag = "PyTorch ResMLP" + (" +new features" if engineered else "")
    best = None; t_tune = time.time(); log = []
    for i, prm in enumerate(ParameterSampler(space, n_iter=10, random_state=RS)):
        pipe = Pipeline(preprocessing(True, engineered) + [("m", TorchMLP(n_seeds=1, random_state=RS))]).set_params(**prm)
        p = cross_val_predict(pipe, tr[FEATS], y, cv=splits, method="predict_proba")[:, 1]
        ap = average_precision_score(y, p); log.append((prm, ap))
        print(tag, i, round(ap, 4), prm, flush=True)
        if best is None or ap > best[0]: best = (ap, prm)
    tune = time.time() - t_tune
    # final: 5-seed ensemble with the best hyperparameters
    final = Pipeline(preprocessing(True, engineered) + [("m", TorchMLP(n_seeds=5, random_state=RS))]).set_params(**best[1])
    p = cross_val_predict(final, tr[FEATS], y, cv=splits, method="predict_proba")[:, 1]
    t0 = time.time(); final.fit(tr[FEATS], y); fit = time.time() - t0
    m = all_metrics(y, p)
    m["PR-AUC fold SD"] = float(np.std([average_precision_score(y[v], p[v]) for _, v in splits]))
    m.update({"Configs tried": 10, "Tuning time (s)": tune, "Fit time (s)": fit})
    print(f"{tag}: PR-AUC {m['PR-AUC']:.4f} R@P95 {m['Recall@P95']:.3f} Brier {m['Brier']:.4f} fit {fit:.0f}s", flush=True)
    out[tag] = dict(name=tag, metrics=m, oof=p, model=final, params=best[1], search_log=log)
    pickle.dump(out, open("cache/s02_torch.pkl", "wb"))
print("DONE")


PyTorch ResMLP +new features 0 0.933 {'m__width': 64, 'm__weight_decay': 0.01, 'm__n_blocks': 2, 'm__lr': 0.01, 'm__epochs': 80, 'm__dropout': 0.0}


PyTorch ResMLP +new features 1 0.9576 {'m__width': 64, 'm__weight_decay': 0.0001, 'm__n_blocks': 1, 'm__lr': 0.01, 'm__epochs': 40, 'm__dropout': 0.3}


PyTorch ResMLP +new features 2 0.9456 {'m__width': 256, 'm__weight_decay': 0.1, 'm__n_blocks': 2, 'm__lr': 0.01, 'm__epochs': 80, 'm__dropout': 0.2}


PyTorch ResMLP +new features 3 0.9569 {'m__width': 64, 'm__weight_decay': 0.1, 'm__n_blocks': 2, 'm__lr': 0.01, 'm__epochs': 40, 'm__dropout': 0.1}


PyTorch ResMLP +new features 4 0.9503 {'m__width': 256, 'm__weight_decay': 0.1, 'm__n_blocks': 2, 'm__lr': 0.001, 'm__epochs': 40, 'm__dropout': 0.0}


PyTorch ResMLP +new features 5 0.9528 {'m__width': 256, 'm__weight_decay': 0.1, 'm__n_blocks': 1, 'm__lr': 0.01, 'm__epochs': 40, 'm__dropout': 0.1}


... (2 outputs left out)

PyTorch ResMLP +new features 8 0.9508 {'m__width': 256, 'm__weight_decay': 0.001, 'm__n_blocks': 2, 'm__lr': 0.003, 'm__epochs': 40, 'm__dropout': 0.2}


PyTorch ResMLP +new features 9 0.945 {'m__width': 128, 'm__weight_decay': 0.0001, 'm__n_blocks': 2, 'm__lr': 0.01, 'm__epochs': 80, 'm__dropout': 0.2}


PyTorch ResMLP +new features: PR-AUC 0.9593 R@P95 0.736 Brier 0.0517 fit 6s


DONE


## 3. Model selection, failure analysis, fixes, SHAP, complexity
Selects the primary model by validation PR-AUC; subgroup failure analysis; threshold / calibration / re-weighting fixes (cross-fitted); SHAP; complexity table. **Validation only.**

In [6]:
"""Step 3: model selection (validation only), failure analysis, failure fixes, SHAP, complexity.
The held-out test set is NOT used anywhere in this script."""
import sys, pickle, time, json, os, io
sys.path.insert(0, ".")
import __main__
from common import *
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib, shap
from sklearn.model_selection import cross_val_predict, StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier
from torch_model import TorchMLP

FIG = "../results/figures"; os.makedirs(FIG, exist_ok=True)
df, lab, cand, tr, te, y, splits = load()
groups = tr["kepid"].values
S1 = pickle.load(open("cache/s01_classical.pkl", "rb")); S2 = pickle.load(open("cache/s02_torch.pkl", "rb"))
R = {**S1, **S2}                                    # name -> dict(metrics, oof, model, params)
R["Ensemble XGB + ResMLP +new features"] = dict(name="Ensemble XGB + ResMLP +new features",
    oof=(R["XGBoost +new features"]["oof"] + R["PyTorch ResMLP +new features"]["oof"]) / 2,
    metrics=None, model=None, params=None)
R["Ensemble XGB + ResMLP +new features"]["metrics"] = all_metrics(y, R["Ensemble XGB + ResMLP +new features"]["oof"])

# ------------------------------------------------------------------ 1. validation table
rows = {k: {**{m: v for m, v in r["metrics"].items()}} for k, r in R.items()}
val = pd.DataFrame(rows).T
val.to_csv("../results/validation_results_final.csv")
print(val[["PR-AUC", "ROC-AUC", "F1", "Recall@P95", "Brier"]].round(4).to_string())

# frozen primary model: highest validation PR-AUC among the candidates
cands = ["XGBoost +new features", "PyTorch ResMLP +new features", "Ensemble XGB + ResMLP +new features"]
primary = max(cands, key=lambda k: R[k]["metrics"]["PR-AUC"])
print("\nPRIMARY MODEL (by validation PR-AUC):", primary)
p0 = R[primary]["oof"]

# ------------------------------------------------------------------ 2. failure analysis (primary, threshold 0.5)
X12 = tr[FEATS]
pred = (p0 >= 0.5).astype(int)
fa = X12.copy(); fa["y"], fa["p"] = y, p0
fa["err"] = (pred != y).astype(int); fa["FP"] = ((y == 0) & (pred == 1)).astype(int); fa["FN"] = ((y == 1) & (pred == 0)).astype(int)
fa["SNR"] = pd.cut(fa.koi_model_snr, [0, 10, 20, 50, 1e9], labels=["<10", "10-20", "20-50", ">50"])
fa["Planet radius (R_earth)"] = pd.cut(fa.koi_prad, [0, 1.25, 2, 4, 15, 1e9], labels=["<1.25", "1.25-2", "2-4", "4-15", ">15"])
fa["Star Teff"] = pd.cut(fa.koi_steff, [0, 4000, 5200, 6000, 6800, 1e9], labels=["<4000K", "4000-5200", "5200-6000", "6000-6800", ">6800K"])
fa["Impact parameter"] = pd.cut(fa.koi_impact, [-1, 0.5, 0.9, 1.0, 1e9], labels=["<0.5", "0.5-0.9", "0.9-1.0", ">1 (grazing)"])
fa["Kepler mag"] = pd.cut(fa.koi_kepmag, [0, 12, 14, 15, 30], labels=["<12 (bright)", "12-14", "14-15", ">15 (faint)"])
fa["Period (d)"] = pd.cut(fa.koi_period, [0, 3, 10, 50, 1e9], labels=["<3", "3-10", "10-50", ">50"])
fa["Missing value"] = np.where(X12.isna().any(axis=1), "has missing", "complete")
def subgroup_table(col):
    g = fa.groupby(col, observed=True)
    out = pd.DataFrame({"n": g.size(), "planet rate": g.y.mean(), "error rate": g.err.mean(), "FP": g.FP.sum(), "FN": g.FN.sum()})
    out["PR-AUC"] = g.apply(lambda d: average_precision_score(d.y, d.p) if 0 < d.y.sum() < len(d) else np.nan)
    return out.round(3)
tabs = {}
print(f"\nFAILURES: total {len(fa)}, errors {fa.err.sum()} ({fa.err.mean():.3f}), false alarms {fa.FP.sum()}, missed planets {fa.FN.sum()}")
for col in ["SNR", "Planet radius (R_earth)", "Star Teff", "Impact parameter", "Kepler mag", "Period (d)", "Missing value"]:
    tabs[col] = subgroup_table(col); print("\n==", col); print(tabs[col].to_string())
pickle.dump(tabs, open("cache/s03_subgroups.pkl", "wb"))
cols = ["koi_prad", "koi_depth", "koi_model_snr", "koi_impact", "koi_period", "koi_duration", "koi_steff"]
prof = pd.DataFrame({k: fa.loc[m, cols].median() for k, m in {
    "False alarm (FP)": fa.FP == 1, "Missed planet (FN)": fa.FN == 1,
    "Found planet (TP)": (fa.y == 1) & (pred == 1), "Rejected FP (TN)": (fa.y == 0) & (pred == 0)}.items()}).T.round(2)
print("\nMedian profile per outcome:\n", prof.to_string())
conf = fa.assign(kepoi_name=tr["kepoi_name"].values).query("err == 1").assign(conf=lambda d: np.abs(d.p - 0.5)).sort_values("conf", ascending=False)
print("\nMost confident mistakes:\n", conf[["kepoi_name", "y", "p", "koi_model_snr", "koi_prad", "koi_impact", "koi_period"]].head(12).round(3).to_string(index=False))
conf.head(30).to_csv("../results/most_confident_errors.csv", index=False)

# ------------------------------------------------------------------ 3. fixes (all evaluated on validation folds only)
def cross_fit(fn, p):
    """Exploratory correction using a shared OOF pool; base fits can include the target fold."""
    out = np.zeros_like(p)
    for a, b in splits:
        out[b] = fn(p[a], y[a], p[b])
    return out
def platt(pa, ya, pb):
    lr = LogisticRegression(C=1e6).fit(np.log(np.clip(pa, 1e-4, 1 - 1e-4) / (1 - np.clip(pa, 1e-4, 1 - 1e-4)))[:, None], ya)
    return lr.predict_proba(np.log(np.clip(pb, 1e-4, 1 - 1e-4) / (1 - np.clip(pb, 1e-4, 1 - 1e-4)))[:, None])[:, 1]
def isotonic(pa, ya, pb):
    return IsotonicRegression(out_of_bounds="clip").fit(pa, ya).predict(pb)
def ece(yt, p, bins=10):
    q = pd.qcut(p, bins, duplicates="drop"); d = pd.DataFrame({"y": yt, "p": p, "q": q}).groupby("q", observed=True)
    return float(sum(len(g) * abs(g.y.mean() - g.p.mean()) for _, g in d) / len(p))
def best_f1_threshold(pa, ya):
    prec, rec, thr = precision_recall_curve(ya, pa); f1 = 2 * prec * rec / (prec + rec + 1e-12)
    return thr[np.argmax(f1[:-1])]
def thr_pred(pa_all=None):
    out = np.zeros(len(y), dtype=int)
    for a, b in splits:
        out[b] = (p0[b] >= best_f1_threshold(p0[a], y[a])).astype(int)
    return out
def row(label, p, hard=None):
    d = {"fix": label, "PR-AUC": average_precision_score(y, p), "Recall@P95": recall_at_precision(y, p),
         "Brier": brier_score_loss(y, p), "ECE": ece(y, p)}
    h = hard if hard is not None else (p >= 0.5)
    d["F1"] = f1_score(y, h); d["errors"] = int((h != y).sum())
    for nm, mask in [("err SNR 10-20", (fa.SNR == "10-20").values), ("err radius>15", (fa["Planet radius (R_earth)"] == ">15").values),
                     ("err missing", (fa["Missing value"] == "has missing").values)]:
        d[nm] = float((h[mask] != y[mask]).mean())
    return d
fixes = [row("baseline (primary, thr 0.5)", p0)]
fixes.append(row("threshold tuned for F1 (cross-fit)", p0, hard=thr_pred()))
fixes.append(row("Platt calibration (cross-fit)", cross_fit(platt, p0)))
fixes.append(row("isotonic calibration (cross-fit)", cross_fit(isotonic, p0)))
# re-weighting: retrain the XGBoost with sample weights, same hyperparameters, same folds
xgbp = R["XGBoost +new features"]["model"]
def xgb_weighted(w, label):
    pw = cross_val_predict(xgbp, X12, y, cv=splits, method="predict_proba", params={"m__sample_weight": w})[:, 1]
    return pw, row(label, pw)
snr = tr.koi_model_snr.values; rad = tr.koi_prad.values
pw_list = {}
for label, w in {
    "re-weight: weak signals (SNR<20) x3": np.where(snr < 20, 3.0, 1.0),
    "re-weight: weak signals x3 + giant planets x3": np.where(snr < 20, 3.0, 1.0) * np.where(rad > 15, 3.0, 1.0),
    "re-weight: inverse class frequency": np.where(y == 1, (y == 0).sum() / (y == 1).sum(), 1.0),
}.items():
    pw, r = xgb_weighted(w, label); fixes.append(r); pw_list[label] = pw
fixdf = pd.DataFrame(fixes).set_index("fix")
print("\nFIXES (validation, 5-fold grouped, cross-fitted):\n", fixdf.round(4).to_string())
fixdf.to_csv("../results/failure_fixes.csv")
pickle.dump({"pw": pw_list}, open("cache/s03_fixes.pkl", "wb"))

fig, ax = plt.subplots(figsize=(5, 5))
for lbl, pp in [("uncalibrated", p0), ("Platt", cross_fit(platt, p0)), ("isotonic", cross_fit(isotonic, p0))]:
    fr, mp = calibration_curve(y, pp, n_bins=10, strategy="quantile"); ax.plot(mp, fr, marker="o", label=lbl)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect"); ax.set_xlabel("Predicted probability"); ax.set_ylabel("Observed fraction of planets")
ax.set_title("Calibration before / after correction"); ax.legend(); plt.tight_layout(); plt.savefig(f"{FIG}/fig_calibration_fix.png", dpi=200); plt.close()

# ------------------------------------------------------------------ 4. SHAP on the XGBoost model (+ permutation importance)
xg = R["XGBoost +new features"]["model"]
Xt = xg[:-1].transform(tr[FEATS]); Xt = pd.DataFrame(Xt, columns=ALL_FEATS)
expl = shap.TreeExplainer(xg.named_steps["m"]); sv = expl.shap_values(Xt)
imp = pd.Series(np.abs(sv).mean(0), index=ALL_FEATS).sort_values(ascending=False)
print("\nMean |SHAP| (log-odds):\n", imp.round(3).to_string()); imp.to_csv("../results/shap_importance.csv")
shap.summary_plot(sv, Xt, show=False, max_display=17); plt.tight_layout(); plt.savefig(f"{FIG}/fig_shap_summary.png", dpi=200, bbox_inches="tight"); plt.close()
fig, ax = plt.subplots(figsize=(6, 5)); imp.iloc[::-1].plot.barh(ax=ax, color=["#2a78d6" if f in NEW_FEATS else "#c4c3b8" for f in imp.index[::-1]])
ax.set_xlabel("mean |SHAP value| (log-odds)"); ax.set_title("Feature importance (blue = new features)"); plt.tight_layout(); plt.savefig(f"{FIG}/fig_shap_bar.png", dpi=200); plt.close()
fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
for a, f in zip(axs, [imp.index[0], imp.index[1], imp.index[2]]):
    shap.dependence_plot(f, sv, Xt, ax=a, show=False, interaction_index=None)
plt.tight_layout(); plt.savefig(f"{FIG}/fig_shap_dependence.png", dpi=200); plt.close()
# permutation importance (PR-AUC drop) on out-of-fold style: use 5 repeats on the full fitted model
from sklearn.inspection import permutation_importance
perm = permutation_importance(xg, tr[FEATS], y, scoring="average_precision", n_repeats=5, random_state=RS, n_jobs=2)
# permuting raw columns permutes the 12 base inputs; engineered columns are derived from them
pimp = pd.Series(perm.importances_mean, index=FEATS).sort_values(ascending=False)
print("\nPermutation importance of the 12 raw inputs (PR-AUC drop):\n", pimp.round(3).to_string()); pimp.to_csv("../results/permutation_importance.csv")
# SHAP of the most confident false alarm
i = np.where((fa.FP.values == 1))[0]; i = i[np.argmax(p0[i])]
print("\nMost confident false alarm:", tr.kepoi_name.values[i], "p=", round(float(p0[i]), 3))
print(pd.Series(sv[i], index=ALL_FEATS).sort_values(key=np.abs, ascending=False).head(6).round(2).to_string())

# ------------------------------------------------------------------ 5. complexity table
def size_kb(m):
    b = io.BytesIO(); joblib.dump(m, b); return len(b.getvalue()) / 1024
def n_par(name, m):
    mm = m.named_steps["m"]
    if "Logistic" in name: return f"{mm.coef_.size + mm.intercept_.size} weights"
    if "Random forest" in name: return f"{len(mm.estimators_)} trees (depth<={mm.max_depth})"
    if "XGBoost" in name: return f"{mm.get_booster().num_boosted_rounds()} trees (depth {mm.max_depth})"
    if "sklearn" in name: return f"{sum(w.size for w in mm.coefs_) + sum(b.size for b in mm.intercepts_)} weights"
    if "PyTorch" in name: return f"{mm.n_parameters()} weights ({len(mm.nets_)} nets)"
interp = {"Logistic regression": "High: one coefficient per feature", "Random forest": "Medium: feature importance only",
          "XGBoost": "Medium-high: TreeSHAP gives exact per-object explanations", "MLP (sklearn)": "Low: black box (post-hoc SHAP only)",
          "PyTorch ResMLP": "Low: black box (post-hoc SHAP only)", "Ensemble": "Low: black box"}
crow = []
for k in ["Logistic regression +new features", "Random forest +new features", "XGBoost +new features", "MLP (sklearn) +new features", "PyTorch ResMLP +new features"]:
    m = R[k]["model"]; Xb = tr[FEATS]
    t0 = time.time()
    for _ in range(5): m.predict_proba(Xb)
    us = (time.time() - t0) / 5 / len(Xb) * 1e6
    crow.append({"Model": k.replace(" +new features", ""), "PR-AUC": R[k]["metrics"]["PR-AUC"], "Size": n_par(k, m), "File size (KB)": size_kb(m),
                 "Tuning time (s)": R[k]["metrics"]["Tuning time (s)"], "Fit time (s)": R[k]["metrics"]["Fit time (s)"],
                 "Inference (us/object)": us, "Interpretability": interp[k.replace(" +new features", "")]})
ens_fit = R["XGBoost +new features"]["metrics"]["Fit time (s)"] + R["PyTorch ResMLP +new features"]["metrics"]["Fit time (s)"]
crow.append({"Model": "Ensemble XGB + ResMLP", "PR-AUC": R["Ensemble XGB + ResMLP +new features"]["metrics"]["PR-AUC"], "Size": "both models",
             "File size (KB)": crow[2]["File size (KB)"] + crow[4]["File size (KB)"], "Tuning time (s)": crow[2]["Tuning time (s)"] + crow[4]["Tuning time (s)"],
             "Fit time (s)": ens_fit, "Inference (us/object)": crow[2]["Inference (us/object)"] + crow[4]["Inference (us/object)"], "Interpretability": interp["Ensemble"]})
cx = pd.DataFrame(crow).set_index("Model"); print("\nCOMPLEXITY:\n", cx.round(2).to_string()); cx.to_csv("../results/complexity.csv")
json.dump({"primary": primary}, open("cache/primary_choice.json", "w"))
print("DONE")


                                     PR-AUC  ROC-AUC      F1  Recall@P95   Brier
Logistic regression                  0.7258   0.8543  0.7148      0.0000  0.1592
Random forest                        0.9433   0.9708  0.8824      0.6506  0.0628
XGBoost                              0.9550   0.9764  0.9004      0.7413  0.0541
MLP (sklearn)                        0.9516   0.9739  0.8880      0.6927  0.0588
Logistic regression +new features    0.7374   0.8592  0.7189      0.0042  0.1562
Random forest +new features          0.9545   0.9764  0.8969      0.7219  0.0562
XGBoost +new features                0.9607   0.9789  0.9045      0.7691  0.0516
MLP (sklearn) +new features          0.9544   0.9752  0.8941      0.7210  0.0562
PyTorch ResMLP +new features         0.9593   0.9778  0.9058      0.7358  0.0517
Ensemble XGB + ResMLP +new features  0.9648   0.9809  0.9084      0.7932  0.0489

PRIMARY MODEL (by validation PR-AUC): Ensemble XGB + ResMLP +new features

FAILURES: total 6041, errors 399 


FIXES (validation, 5-fold grouped, cross-fitted):
                                                PR-AUC  Recall@P95   Brier     ECE      F1  errors  err SNR 10-20  err radius>15  err missing
fix                                                                                                                                         
baseline (primary, thr 0.5)                    0.9648      0.7932  0.0489  0.0065  0.9084     399         0.1212         0.0097       0.0244
threshold tuned for F1 (cross-fit)             0.9648      0.7932  0.0489  0.0065  0.9109     400         0.1241         0.0103       0.0341
Platt calibration (cross-fit)                  0.9644      0.7950  0.0489  0.0047  0.9092     397         0.1198         0.0097       0.0293
isotonic calibration (cross-fit)               0.9616      0.7857  0.0491  0.0033  0.9087     405         0.1241         0.0097       0.0244
re-weight: weak signals (SNR<20) x3            0.9597      0.7862  0.0524  0.0088  0.9029     420     


Mean |SHAP| (log-odds):
 koi_prad             0.830
star_density         0.740
koi_model_snr        0.643
depth_consistency    0.521
koi_period           0.519
duration_ratio       0.313
koi_insol            0.299
koi_steff            0.280
koi_teq              0.254
rp_over_rs           0.251
koi_depth            0.228
koi_duration         0.179
koi_impact           0.152
koi_srad             0.146
koi_slogg            0.102
implied_noise        0.102
koi_kepmag           0.079



Permutation importance of the 12 raw inputs (PR-AUC drop):
 koi_prad         0.167
koi_period       0.100
koi_duration     0.088
koi_model_snr    0.070
koi_srad         0.026
koi_depth        0.011
koi_slogg        0.010
koi_steff        0.007
koi_insol        0.002
koi_impact       0.002
koi_kepmag       0.001
koi_teq          0.001

Most confident false alarm: K02050.01 p= 0.99
koi_model_snr        0.63
koi_prad             0.46
koi_kepmag          -0.42
duration_ratio      -0.40
star_density         0.36
depth_consistency    0.30



COMPLEXITY:
                        PR-AUC                     Size  File size (KB)  Tuning time (s)  Fit time (s)  Inference (us/object)                                           Interpretability
Model                                                                                                                                                                                  
Logistic regression      0.74               18 weights            3.28             0.58          0.04                   1.06                          High: one coefficient per feature
Random forest            0.95  600 trees (depth<=None)        24416.66           132.68          5.03                  31.49                            Medium: feature importance only
XGBoost                  0.96      400 trees (depth 8)         1831.92            35.19          0.65                   5.63  Medium-high: TreeSHAP gives exact per-object explanations
MLP (sklearn)            0.95             1217 weights           3

## 3b. Dataset statistics and leakage ablation
Disposition counts, missingness, and what happens when the vetting columns or process signals are kept. **Validation only.**

In [7]:
"""Step 3b: dataset statistics and leakage ablation (train/validation part only; test set untouched)."""
import sys, pickle, json
sys.path.insert(0, ".")
from common import *
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
df, lab, cand, tr, te, y, splits = load()
S1 = pickle.load(open("cache/s01_classical.pkl", "rb"))
print("Disposition counts:", df.koi_disposition.value_counts().to_dict(), "| total", len(df), "| columns", df.shape[1])
print("Labelled:", len(lab), "positive rate", round(lab.y.mean(), 4), "| train", len(tr), tr.kepid.nunique(), "stars | test", len(te), te.kepid.nunique(), "stars")
print("Stars with >1 KOI:", int((df.kepid.value_counts() > 1).sum()), "of", df.kepid.nunique())
print("\nMissing rate per base feature (labelled):\n", lab[FEATS].isna().mean().round(3).to_string())
print("Rows with >=1 missing:", round(lab[FEATS].isna().any(axis=1).mean(), 4), "| planet rate among them:", round(lab[lab[FEATS].isna().any(axis=1)].y.mean(), 3))
# leakage ablation
FLAGS = ["koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec"]
leaky = FEATS + ["koi_score"] + FLAGS
xg = S1["XGBoost"]["model"]
mk = lambda: clone(xg)
p_base = S1["XGBoost"]["oof"]
pl = cross_val_predict(mk(), tr[leaky], y, cv=splits, method="predict_proba")[:, 1]
print("\nXGB honest 12 feats      PR-AUC", round(average_precision_score(y, p_base), 4))
print("XGB + koi_score + 4 flags PR-AUC", round(average_precision_score(y, pl), 4))
fl = Pipeline([("imp", SimpleImputer(strategy="median")), ("m", LogisticRegression(max_iter=2000))])
pf = cross_val_predict(fl, tr[FLAGS], y, cv=splits, method="predict_proba")[:, 1]
print("Logistic regression on the 4 flags only PR-AUC", round(average_precision_score(y, pf), 4))
ps = cross_val_predict(Pipeline([("imp", SimpleImputer(strategy="median")), ("m", LogisticRegression(max_iter=2000))]), tr[["koi_score"]], y, cv=splits, method="predict_proba")[:, 1]
print("Logistic regression on koi_score only PR-AUC", round(average_precision_score(y, ps), 4))
# process signals
tr2 = tr.copy(); tr2["n_koi_system"] = tr2.kepid.map(df.kepid.value_counts())
pp = cross_val_predict(mk(), tr2[FEATS + ["koi_tce_plnt_num", "n_koi_system"]], y, cv=splits, method="predict_proba")[:, 1]
print("XGB (same hp) + planet number + KOIs per star PR-AUC", round(average_precision_score(y, pp), 4))
multi = df.kepid.map(df.kepid.value_counts()) > 1
print("\nPlanet share by multi-KOI star:\n", pd.crosstab(multi.rename(">1 KOI"), df.koi_disposition, normalize="index").round(3).to_string())
# EDA stats for report
print("\nPlanet rate by SNR bin:", tr.groupby(pd.cut(tr.koi_model_snr, [0, 10, 20, 50, 1e9]), observed=True).y.mean().round(3).to_dict())
print("Planet rate by radius bin:", tr.groupby(pd.cut(tr.koi_prad, [0, 2, 4, 15, 1e9]), observed=True).y.mean().round(3).to_dict())
# ablation figure
fig, ax = plt.subplots(figsize=(6.5, 3.6))
vals = {"12 physical features\n(honest)": average_precision_score(y, p_base), "+ planet number,\nKOIs per star": average_precision_score(y, pp),
        "+ koi_score + 4 flags\n(leaky)": average_precision_score(y, pl)}
ax.bar(vals.keys(), vals.values(), color=["#2a78d6", "#c4c3b8", "#d6402a"]); ax.set_ylim(0.9, 1.0); ax.set_ylabel("Validation PR-AUC (XGBoost)")
for i, v in enumerate(vals.values()): ax.text(i, v + 0.001, f"{v:.3f}", ha="center")
plt.tight_layout(); plt.savefig("../results/figures/fig_leakage.png", dpi=200)
json.dump({"honest": float(vals["12 physical features\n(honest)"]), "process": float(vals["+ planet number,\nKOIs per star"]), "leaky": float(vals["+ koi_score + 4 flags\n(leaky)"]),
           "flags_only": float(average_precision_score(y, pf)), "score_only": float(average_precision_score(y, ps))}, open("../results/leakage_ablation.json", "w"))


Disposition counts: {'FALSE POSITIVE': 4839, 'CONFIRMED': 2748, 'CANDIDATE': 1977} | total 9564 | columns 49
Labelled: 7587 positive rate 0.3622 | train 6041 5312 stars | test 1546 1329 stars
Stars with >1 KOI: 938 of 8214

Missing rate per base feature (labelled):
 koi_period       0.000
koi_duration     0.000
koi_depth        0.034
koi_impact       0.034
koi_model_snr    0.034
koi_prad         0.034
koi_teq          0.034
koi_insol        0.029
koi_steff        0.034
koi_slogg        0.034
koi_srad         0.034
koi_kepmag       0.000
Rows with >=1 missing: 0.0343 | planet rate among them: 0.008



XGB honest 12 feats      PR-AUC 0.955
XGB + koi_score + 4 flags PR-AUC 0.9984
Logistic regression on the 4 flags only PR-AUC 0.9603
Logistic regression on koi_score only PR-AUC 0.9604


XGB (same hp) + planet number + KOIs per star PR-AUC 0.9652

Planet share by multi-KOI star:
 koi_disposition  CANDIDATE  CONFIRMED  FALSE POSITIVE
>1 KOI                                               
False                0.206      0.176           0.618
True                 0.208      0.641           0.151

Planet rate by SNR bin: {Interval(0.0, 10.0, closed='right'): 0.054, Interval(10.0, 20.0, closed='right'): 0.401, Interval(20.0, 50.0, closed='right'): 0.612, Interval(50.0, 1000000000.0, closed='right'): 0.26}
Planet rate by radius bin: {Interval(0.0, 2.0, closed='right'): 0.44, Interval(2.0, 4.0, closed='right'): 0.692, Interval(4.0, 15.0, closed='right'): 0.422, Interval(15.0, 1000000000.0, closed='right'): 0.011}


## 4. Freeze → one test evaluation → CANDIDATE ranking
The frozen configuration is written to `results/frozen_config.json` *before* the test set is scored. Models are then refit on all labelled data to rank the 1,977 CANDIDATE objects.

In [8]:
"""Step 4: freeze everything on validation data, evaluate ONCE on the held-out test set, rank the CANDIDATE objects."""
import sys, pickle, json, os, time
sys.path.insert(0, ".")
from common import *
import joblib
from sklearn.base import clone
from scipy.stats import spearmanr

df, lab, cand, tr, te, y, splits = load()
S1 = pickle.load(open("cache/s01_classical.pkl", "rb")); S2 = pickle.load(open("cache/s02_torch.pkl", "rb"))
R = {**S1, **S2}
xgb_n, nn_n = "XGBoost +new features", "PyTorch ResMLP +new features"
p_val = (R[xgb_n]["oof"] + R[nn_n]["oof"]) / 2

# ---------- freeze (validation only) ----------
prec, rec, thr = precision_recall_curve(y, p_val)
f1 = 2 * prec * rec / (prec + rec + 1e-12)
thr_f1 = float(thr[np.argmax(f1[:-1])])
ok = np.where(prec[:-1] >= 0.95)[0]; thr_p95 = float(thr[ok[np.argmax(rec[ok])]])
frozen = {"primary_model": "Equal-weight average of XGBoost and PyTorch ResMLP (17 features)",
          "features": ALL_FEATS, "threshold_default": 0.5, "threshold_F1_optimal_on_validation": thr_f1,
          "threshold_95pct_precision_on_validation": thr_p95, "val_PR_AUC": float(average_precision_score(y, p_val)),
          "xgb_params": R[xgb_n]["params"], "resmlp_params": R[nn_n]["params"], "split_seed": RS,
          "selection_rule": "highest 5-fold grouped-CV PR-AUC on the 80% train/validation part; test set not used"}
os.makedirs("../results", exist_ok=True); json.dump(frozen, open("../results/frozen_config.json", "w"), indent=2, default=str)
print("FROZEN:", json.dumps({k: v for k, v in frozen.items() if k.startswith(("thr", "threshold", "val"))}, default=str))

# ---------- ONE evaluation on the held-out test set ----------
yt = te["y"].values; Xte = te[FEATS]
tp = {k: R[k]["model"].predict_proba(Xte)[:, 1] for k in R if R[k]["model"] is not None}
tp["Ensemble XGB + ResMLP +new features"] = (tp[xgb_n] + tp[nn_n]) / 2
rows = {}
for k, p in tp.items():
    m = all_metrics(yt, p); rows[k] = m
test_df = pd.DataFrame(rows).T; test_df.to_csv("../results/test_results.csv")
print("\nTEST RESULTS (single evaluation):\n", test_df.round(4).to_string())

P = tp["Ensemble XGB + ResMLP +new features"]
# thresholds chosen on validation, applied to test
from sklearn.metrics import precision_score, recall_score, confusion_matrix
tr_rows = []
for nm, t in [("default 0.5", 0.5), ("F1-optimal (val)", thr_f1), ("95%-precision (val)", thr_p95)]:
    h = (P >= t).astype(int); tn, fp, fn, tpp = confusion_matrix(yt, h).ravel()
    tr_rows.append({"threshold": nm, "value": round(t, 3), "precision": precision_score(yt, h), "recall": recall_score(yt, h), "F1": f1_score(yt, h), "TP": tpp, "FP": fp, "FN": fn, "TN": tn})
thr_df = pd.DataFrame(tr_rows).set_index("threshold"); thr_df.to_csv("../results/test_thresholds.csv"); print("\nThresholds on test:\n", thr_df.round(3).to_string())

# bootstrap CI, resampling whole stars (so KOIs of the same star stay together)
rng = np.random.default_rng(RS); stars = te["kepid"].values; ustars = np.unique(stars)
idx_by = {s: np.where(stars == s)[0] for s in ustars}; B = []
for _ in range(1000):
    pick = rng.choice(ustars, len(ustars), replace=True); ii = np.concatenate([idx_by[s] for s in pick])
    if yt[ii].sum() == 0: continue
    B.append([average_precision_score(yt[ii], P[ii]), roc_auc_score(yt[ii], P[ii]), f1_score(yt[ii], P[ii] >= .5), recall_at_precision(yt[ii], P[ii]), brier_score_loss(yt[ii], P[ii])])
B = np.array(B); ci = pd.DataFrame({"2.5%": np.percentile(B, 2.5, 0), "97.5%": np.percentile(B, 97.5, 0)}, index=["PR-AUC", "ROC-AUC", "F1", "Recall@P95", "Brier"])
ci.to_csv("../results/test_bootstrap_ci.csv"); print("\nBootstrap 95% CI (primary):\n", ci.round(4).to_string())

# test-set failure breakdown
fa = Xte.copy(); fa["y"], fa["p"] = yt, P; fa["err"] = ((P >= .5).astype(int) != yt).astype(int)
fa["SNR"] = pd.cut(fa.koi_model_snr, [0, 10, 20, 50, 1e9], labels=["<10", "10-20", "20-50", ">50"])
fa["Radius"] = pd.cut(fa.koi_prad, [0, 1.25, 2, 4, 15, 1e9], labels=["<1.25", "1.25-2", "2-4", "4-15", ">15"])
fa["Missing"] = np.where(Xte.isna().any(axis=1), "has missing", "complete")
tt = {}
for c in ["SNR", "Radius", "Missing"]:
    g = fa.groupby(c, observed=True); tt[c] = pd.DataFrame({"n": g.size(), "planet rate": g.y.mean(), "error rate": g.err.mean()}).round(3); print("\n", tt[c].to_string())
print("test errors:", int(fa.err.sum()), "of", len(fa))
pickle.dump({"test_df": test_df, "thr_df": thr_df, "ci": ci, "tt": tt, "P": P, "yt": yt}, open("cache/s04_test.pkl", "wb"))

# ---------- refit on ALL labelled data and rank the CANDIDATEs ----------
Xall, yall = lab[FEATS], lab["y"].values
xg_f = clone(R[xgb_n]["model"]).fit(Xall, yall)
nn_f = clone(R[nn_n]["model"]).fit(Xall, yall)
joblib.dump(xg_f, "../models/final_xgboost.joblib"); joblib.dump(nn_f, "../models/final_resmlp.joblib")
pc = (xg_f.predict_proba(cand[FEATS])[:, 1] + nn_f.predict_proba(cand[FEATS])[:, 1]) / 2
out = cand[["kepoi_name", "kepid", "koi_period", "koi_prad", "koi_model_snr", "koi_steff"]].copy(); out["planet_probability"] = pc
out = out.sort_values("planet_probability", ascending=False); out["rank"] = np.arange(1, len(out) + 1)
out.to_csv("../results/candidate_ranking.csv", index=False)
print("\nCANDIDATES:", len(out), "| p>=0.5:", int((pc >= .5).sum()), "| p>=thr95:", int((pc >= thr_p95).sum()), "| p<0.1:", int((pc < .1).sum()), "| missing-feature rows:", int(cand[FEATS].isna().any(axis=1).sum()))
print(out.head(10).round(3).to_string(index=False))
# external sanity check: agreement with the NASA Robovetter score (NOT used as a feature)
sc = cand["koi_score"].reindex(out.index) if False else cand.loc[out.index, "koi_score"]
m = sc.notna().values; print("Spearman(our probability, koi_score) on candidates:", round(spearmanr(out.planet_probability.values[m], sc.values[m])[0], 3))
print("candidate p-deciles counts:", np.histogram(pc, bins=np.linspace(0, 1, 11))[0].tolist())
json.dump({"n": len(out), "ge05": int((pc >= .5).sum()), "ge_p95": int((pc >= thr_p95).sum()), "lt01": int((pc < .1).sum()),
           "spearman_koi_score": float(spearmanr(out.planet_probability.values[m], sc.values[m])[0]), "hist": np.histogram(pc, bins=np.linspace(0, 1, 11))[0].tolist()},
          open("../results/candidate_summary.json", "w"))
print("DONE")


FROZEN: {"threshold_default": 0.5, "threshold_F1_optimal_on_validation": 0.3555489778518677, "threshold_95pct_precision_on_validation": 0.8068511486053467, "val_PR_AUC": 0.9648336679556053}

TEST RESULTS (single evaluation):
                                      PR-AUC  ROC-AUC      F1  Recall@P95   Brier
Logistic regression                  0.7587   0.8629  0.7423      0.0221  0.1539
Random forest                        0.9581   0.9747  0.9065      0.7819  0.0571
XGBoost                              0.9623   0.9780  0.9075      0.8467  0.0524
MLP (sklearn)                        0.9635   0.9791  0.9142      0.7768  0.0519
Logistic regression +new features    0.7653   0.8667  0.7480      0.0664  0.1514
Random forest +new features          0.9627   0.9788  0.9164      0.8279  0.0506
XGBoost +new features                0.9680   0.9817  0.9224      0.8535  0.0453
MLP (sklearn) +new features          0.9552   0.9759  0.9056      0.7853  0.0549
PyTorch ResMLP +new features         0.9626  


Bootstrap 95% CI (primary):
               2.5%   97.5%
PR-AUC      0.9565  0.9791
ROC-AUC     0.9761  0.9871
F1          0.9025  0.9361
Recall@P95  0.7368  0.9199
Brier       0.0386  0.0547

          n  planet rate  error rate
SNR                                
<10    166        0.024       0.036
10-20  397        0.456       0.101
20-50  403        0.645       0.084
>50    525        0.270       0.025

           n  planet rate  error rate
Radius                              
<1.25   272        0.320       0.096
1.25-2  326        0.589       0.095
2-4     355        0.685       0.062
4-15    120        0.508       0.092
>15     418        0.010       0.007

                 n  planet rate  error rate
Missing                                   
complete     1491        0.394       0.062
has missing    55        0.000       0.000
test errors: 93 of 1546



CANDIDATES: 1977 | p>=0.5: 548 | p>=thr95: 313 | p<0.1: 919 | missing-feature rows: 104
kepoi_name    kepid  koi_period  koi_prad  koi_model_snr  koi_steff  planet_probability  rank
 K01856.01 11570749      46.299      4.33           55.0     5396.0               0.996     1
 K01861.01  3648437      17.504      3.56           47.5     5815.0               0.994     2
 K00865.01  6862328     119.021      8.98          223.5     5453.0               0.993     3
 K01145.01  8313667      30.587      2.32           33.1     5944.0               0.993     4
 K00501.01  4951877      24.796      2.70           43.6     5770.0               0.993     5
 K02009.01  2449431      86.749      2.23           35.7     5579.0               0.992     6
 K00268.01  3425851     110.378      3.02          108.9     6342.0               0.991     7
 K02691.01  4552729      97.446      3.05           62.8     4735.0               0.991     8
 K02984.01  7918652      11.455      1.12           15.8     6231

## 5. Report figures

In [9]:
"""Step 5: report figures from the saved results."""
import sys, pickle, json
sys.path.insert(0, ".")
from common import *
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
FIG = "../results/figures"
val = pd.read_csv("../results/validation_results_final.csv", index_col=0); test = pd.read_csv("../results/test_results.csv", index_col=0)
S = pickle.load(open("cache/s04_test.pkl", "rb")); tabs = pickle.load(open("cache/s03_subgroups.pkl", "rb"))
# 1. validation vs test PR-AUC
order = ["Logistic regression", "Random forest", "XGBoost", "MLP (sklearn)", "Logistic regression +new features", "Random forest +new features",
         "XGBoost +new features", "MLP (sklearn) +new features", "PyTorch ResMLP +new features", "Ensemble XGB + ResMLP +new features"]
short = ["LR", "RF", "XGB", "MLP", "LR+", "RF+", "XGB+", "MLP+", "ResMLP+", "Ens+"]
fig, ax = plt.subplots(figsize=(8, 3.8)); x = np.arange(len(order)); w = 0.38
ax.bar(x - w / 2, val.loc[order, "PR-AUC"], w, label="validation (5-fold grouped CV)", color="#c4c3b8")
ax.bar(x + w / 2, test.loc[order, "PR-AUC"], w, label="held-out test (once)", color="#2a78d6")
ax.set_xticks(x); ax.set_xticklabels(short); ax.set_ylim(0.7, 1.0); ax.set_ylabel("PR-AUC"); ax.legend(loc="lower right")
ax.set_title("PR-AUC by model ('+' = with the 5 new features)"); plt.tight_layout(); plt.savefig(f"{FIG}/fig_models_pr_auc.png", dpi=200); plt.close()
# 2. PR curve on test (primary)
P, yt = S["P"], S["yt"]; pr, rc, _ = precision_recall_curve(yt, P)
fig, ax = plt.subplots(figsize=(4.6, 4.2)); ax.plot(rc, pr, color="#2a78d6", lw=2); ax.axhline(0.95, ls="--", c="grey", lw=1)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title(f"Test precision-recall curve (AP = {average_precision_score(yt, P):.3f})"); ax.set_ylim(0.4, 1.02)
plt.tight_layout(); plt.savefig(f"{FIG}/fig_test_pr_curve.png", dpi=200); plt.close()
# 3. failure analysis: error rate by subgroup (validation)
fig, axs = plt.subplots(1, 3, figsize=(12, 3.6))
for a, (col, title) in zip(axs, [("SNR", "Transit SNR"), ("Planet radius (R_earth)", "Planet radius (Earth radii)"), ("Impact parameter", "Impact parameter")]):
    t = tabs[col]; a.bar(t.index.astype(str), t["error rate"], color="#d6402a"); a.set_title(title); a.set_ylabel("error rate"); a.tick_params(axis="x", labelsize=8)
    for i, (e, n) in enumerate(zip(t["error rate"], t["n"])): a.text(i, e + 0.002, f"n={n}", ha="center", fontsize=7)
plt.tight_layout(); plt.savefig(f"{FIG}/fig_failure_subgroups.png", dpi=200); plt.close()
# 4. candidate probabilities
c = pd.read_csv("../results/candidate_ranking.csv")
fig, ax = plt.subplots(figsize=(5, 3.4)); ax.hist(c.planet_probability, bins=20, color="#2a78d6"); ax.set_xlabel("Predicted probability of being a planet"); ax.set_ylabel("CANDIDATE objects")
ax.set_title("Scores of the 1,977 CANDIDATE objects"); plt.tight_layout(); plt.savefig(f"{FIG}/fig_candidates.png", dpi=200); plt.close()
print("figures saved")


figures saved


## 6. Extra mitigations and interpretability trade-off
Selective deferral to human vetters, a weak-signal specialist model, and the cost of interpretability (small trees, top-k features). **Validation only.**

In [10]:
"""Step 6 (validation only, test set NOT used): extra mitigations and a quantified interpretability trade-off."""
import sys, pickle, json
sys.path.insert(0, ".")
from common import *
from sklearn.model_selection import cross_val_predict
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.base import clone
df, lab, cand, tr, te, y, splits = load()
S1 = pickle.load(open("cache/s01_classical.pkl", "rb")); S2 = pickle.load(open("cache/s02_torch.pkl", "rb"))
xgb_n, nn_n = "XGBoost +new features", "PyTorch ResMLP +new features"
p0 = (S1[xgb_n]["oof"] + S2[nn_n]["oof"]) / 2
snr = tr.koi_model_snr.values
out = {}

# ---- A. selective prediction: defer uncertain objects to human vetters --------------------------
rows = []
for band in [0, 0.05, 0.1, 0.15, 0.2, 0.3]:
    keep = np.abs(p0 - 0.5) >= band
    h = (p0 >= 0.5).astype(int)
    m = keep & (snr >= 10) & (snr < 20)
    rows.append({"defer band |p-0.5|<": band, "coverage": keep.mean(), "error rate on decided": (h[keep] != y[keep]).mean(),
                 "errors caught by deferral": int(((h != y) & ~keep).sum()), "objects deferred": int((~keep).sum()),
                 "error SNR 10-20 (decided)": (h[m] != y[m]).mean(), "coverage SNR 10-20": m.sum() / ((snr >= 10) & (snr < 20)).sum()})
A = pd.DataFrame(rows).set_index("defer band |p-0.5|<"); print("A. SELECTIVE PREDICTION\n", A.round(3).to_string()); A.to_csv("../results/mitigation_deferral.csv")

# ---- B. specialist model for weak signals (SNR<30), blended with the global model ----------------
xg = S1[xgb_n]["model"]; weak = snr < 30
pb = p0.copy()
for a, b in splits:
    a_w = a[weak[a]]; b_w = b[weak[b]]
    sp = clone(xg).fit(tr[FEATS].iloc[a_w], y[a_w])
    pb[b_w] = 0.5 * p0[b_w] + 0.5 * sp.predict_proba(tr[FEATS].iloc[b_w])[:, 1]
def rep(label, p):
    h = (p >= .5).astype(int); m = (snr >= 10) & (snr < 20)
    return {"variant": label, "PR-AUC": average_precision_score(y, p), "Recall@P95": recall_at_precision(y, p), "errors": int((h != y).sum()),
            "error SNR 10-20": float((h[m] != y[m]).mean()), "PR-AUC SNR 10-20": average_precision_score(y[m], p[m])}
B = pd.DataFrame([rep("global ensemble", p0), rep("global + weak-signal specialist (SNR<30)", pb)]).set_index("variant")
print("\nB. WEAK-SIGNAL SPECIALIST\n", B.round(4).to_string()); B.to_csv("../results/mitigation_specialist.csv")

# ---- C. interpretability vs accuracy: how much does readability cost? ---------------------------
rows = []
for d in [2, 3, 4, 6, 8]:
    pipe = Pipeline(preprocessing(False, True) + [("m", DecisionTreeClassifier(max_depth=d, min_samples_leaf=20, random_state=RS))])
    p = cross_val_predict(pipe, tr[FEATS], y, cv=splits, method="predict_proba")[:, 1]
    rows.append({"model": f"decision tree depth {d}", "readable size": f"{2 ** d} leaves", "PR-AUC": average_precision_score(y, p), "Recall@P95": recall_at_precision(y, p)})
# Exploratory: the SHAP ranking and imputer see the entire development partition.
# A fold-independent estimate would fit both again inside each training fold.
imp = pd.read_csv("../results/shap_importance.csv", index_col=0).iloc[:, 0]
for k in [3, 5, 8]:
    top = list(imp.index[:k]); raw = [f for f in top if f in FEATS]
    # engineered features need raw columns: keep raw columns required to build them
    Xk = Pipeline(preprocessing(False, True))   # build full features then select columns
    full = Pipeline(preprocessing(False, True)).fit_transform(tr[FEATS]); full = pd.DataFrame(full, columns=ALL_FEATS)
    pk = cross_val_predict(clone(xg.named_steps["m"]), full[top].values, y, cv=splits, method="predict_proba")[:, 1]
    rows.append({"model": f"XGBoost on top-{k} SHAP features", "readable size": f"{k} features", "PR-AUC": average_precision_score(y, pk), "Recall@P95": recall_at_precision(y, pk)})
rows.append({"model": "XGBoost (17 features)", "readable size": "200 trees", "PR-AUC": average_precision_score(y, S1[xgb_n]["oof"]), "Recall@P95": recall_at_precision(y, S1[xgb_n]["oof"])})
rows.append({"model": "Ensemble XGB + ResMLP", "readable size": "black box", "PR-AUC": average_precision_score(y, p0), "Recall@P95": recall_at_precision(y, p0)})
C = pd.DataFrame(rows).set_index("model"); print("\nC. INTERPRETABILITY vs ACCURACY\n", C.round(4).to_string()); C.to_csv("../results/interpretability_tradeoff.csv")
pipe = Pipeline(preprocessing(False, True) + [("m", DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=RS))]).fit(tr[FEATS], y)
print("\nDepth-3 tree rules:\n", export_text(pipe.named_steps["m"], feature_names=ALL_FEATS, decimals=2))


A. SELECTIVE PREDICTION
                      coverage  error rate on decided  errors caught by deferral  objects deferred  error SNR 10-20 (decided)  coverage SNR 10-20
defer band |p-0.5|<                                                                                                                             
0.00                    1.000                  0.066                          0                 0                      0.120               1.000
0.05                    0.985                  0.061                         38                90                      0.115               0.977
0.10                    0.967                  0.053                         91               200                      0.104               0.944
0.15                    0.947                  0.046                        133               318                      0.090               0.903
0.20                    0.931                  0.040                        173               417        


B. WEAK-SIGNAL SPECIALIST
                                           PR-AUC  Recall@P95  errors  error SNR 10-20  PR-AUC SNR 10-20
variant                                                                                                
global ensemble                           0.9648      0.7932     399           0.1204            0.9123
global + weak-signal specialist (SNR<30)  0.9638      0.7973     405           0.1226            0.9072



C. INTERPRETABILITY vs ACCURACY
                                readable size  PR-AUC  Recall@P95
model                                                           
decision tree depth 2               4 leaves  0.6052      0.0000
decision tree depth 3               8 leaves  0.7034      0.0000
decision tree depth 4              16 leaves  0.7675      0.0000
decision tree depth 6              64 leaves  0.8640      0.0000
decision tree depth 8             256 leaves  0.8906      0.0000
XGBoost on top-3 SHAP features    3 features  0.8463      0.0782
XGBoost on top-5 SHAP features    5 features  0.9399      0.5942
XGBoost on top-8 SHAP features    8 features  0.9539      0.7395
XGBoost (17 features)              200 trees  0.9607      0.7691
Ensemble XGB + ResMLP              black box  0.9648      0.7932

Depth-3 tree rules:
 |--- koi_prad <= 1.13
|   |--- koi_model_snr <= 1.10
|   |   |--- koi_model_snr <= 1.05
|   |   |   |--- class: 0
|   |   |--- koi_model_snr >  1.05
|   |   |   |--

## 7. Sensitivity to the split and evaluation design
Repeated grouped CV, alternative grouped 80/20 splits, and grouped versus row-wise CV. All three use only the development partition.

In [11]:
"""Step 7: how sensitive are the conclusions to the split and the evaluation design?
(A) 5 repeats of grouped 5-fold CV with different fold seeds (train/validation part only);
(B) Grouped 80/20 splits within development of the development partition, frozen hyper-parameters, models refit on each split;
(C) grouped vs. ordinary (ungrouped) cross-validation.
Hyper-parameters are the ones frozen in s01/s02 (no re-tuning). The ORIGINAL held-out test set is not used for any decision here."""
import sys, os, pickle
sys.path.insert(0, ".")
from common import *
from torch_model import TorchMLP
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GroupShuffleSplit, StratifiedGroupKFold

QUICK = os.environ.get("QUICK") == "1"
df, lab, cand, tr, te, y, splits = load()
S1 = pickle.load(open("cache/s01_classical.pkl", "rb")); S2 = pickle.load(open("cache/s02_torch.pkl", "rb"))
M = {"Logistic regression": clone(S1["Logistic regression +new features"]["model"]), "Random forest": clone(S1["Random forest +new features"]["model"]),
     "XGBoost": clone(S1["XGBoost +new features"]["model"]), "ResMLP": clone(S2["PyTorch ResMLP +new features"]["model"])}
def fit_pred(name, Xa, ya, Xb):
    return clone(M[name]).fit(Xa, ya).predict_proba(Xb)[:, 1]
def metrics(yt, P):
    P = dict(P); P["Ensemble XGB+ResMLP"] = (P["XGBoost"] + P["ResMLP"]) / 2
    return {k: (average_precision_score(yt, p), recall_at_precision(yt, p)) for k, p in P.items()}

# ---------------- (A) repeated grouped CV on the train/validation part ----------------
reps = 1 if QUICK else 5; rowsA = []
for r in range(reps):
    cvr = list(StratifiedGroupKFold(5, shuffle=True, random_state=100 + r).split(tr, y, tr["kepid"]))
    oof = {k: np.zeros(len(y)) for k in M}
    for a, b in cvr:
        for k in M: oof[k][b] = fit_pred(k, tr[FEATS].iloc[a], y[a], tr[FEATS].iloc[b])
    for k, (ap, r95) in metrics(y, oof).items(): rowsA.append({"repeat": r, "model": k, "PR-AUC": ap, "Recall@P95": r95})
    print("repeat", r, "done", flush=True)
A = pd.DataFrame(rowsA); A.to_csv("../results/sensitivity_repeated_cv_raw.csv", index=False)
summA = A.groupby("model")[["PR-AUC", "Recall@P95"]].agg(["mean", "std"]); print("\n(A) repeated grouped 5-fold CV\n", summA.round(4).to_string()); summA.to_csv("../results/sensitivity_repeated_cv.csv")
pv = A.pivot(index="repeat", columns="model", values="PR-AUC"); d = pv["Ensemble XGB+ResMLP"] - pv["XGBoost"]
print("Ensemble - XGBoost PR-AUC per repeat:", d.round(4).tolist(), "| mean", round(d.mean(), 4), "| ensemble better in", int((d > 0).sum()), "of", len(d))

# ---------------- (B) many independent outer splits (frozen hyper-parameters) ----------------
nout = 2 if QUICK else 10; rowsB = []
for s in range(nout):
    a, b = next(GroupShuffleSplit(1, test_size=0.2, random_state=1000 + s).split(tr, y, tr["kepid"]))
    la, lb = tr.iloc[a], tr.iloc[b]
    P = {k: fit_pred(k, la[FEATS], la["y"].values, lb[FEATS]) for k in M}
    for k, (ap, r95) in metrics(lb["y"].values, P).items(): rowsB.append({"split": s, "model": k, "PR-AUC": ap, "Recall@P95": r95})
    print("outer split", s, "done", flush=True)
B = pd.DataFrame(rowsB); B.to_csv("../results/sensitivity_outer_splits_raw.csv", index=False)
summB = B.groupby("model")[["PR-AUC", "Recall@P95"]].agg(["mean", "std", "min", "max"]); print("\n(B) Grouped 80/20 splits within development\n", summB.round(4).to_string()); summB.to_csv("../results/sensitivity_outer_splits.csv")
pvB = B.pivot(index="split", columns="model", values="PR-AUC"); dB = pvB["Ensemble XGB+ResMLP"] - pvB["XGBoost"]
print("Ensemble - XGBoost development holdout PR-AUC per split:", dB.round(4).tolist(), "| mean", round(dB.mean(), 4), "| ensemble better in", int((dB > 0).sum()), "of", len(dB))

# ---------------- (C) grouped vs ungrouped CV (XGBoost, same hyper-parameters) ----------------
ung = np.zeros(len(y)); grp = np.zeros(len(y))
for a, b in StratifiedKFold(5, shuffle=True, random_state=RS).split(tr, y):
    ung[b] = fit_pred("XGBoost", tr[FEATS].iloc[a], y[a], tr[FEATS].iloc[b])
for a, b in splits: grp[b] = fit_pred("XGBoost", tr[FEATS].iloc[a], y[a], tr[FEATS].iloc[b])
C = pd.DataFrame({"ungrouped 5-fold CV": [average_precision_score(y, ung), recall_at_precision(y, ung)], "grouped by star 5-fold CV": [average_precision_score(y, grp), recall_at_precision(y, grp)]}, index=["PR-AUC", "Recall@P95"])
print("\n(C) grouped vs ungrouped\n", C.round(4).to_string()); C.to_csv("../results/sensitivity_grouping.csv")


repeat 0 done


repeat 1 done


repeat 2 done


repeat 3 done


repeat 4 done



(A) repeated grouped 5-fold CV
                      PR-AUC         Recall@P95        
                       mean     std       mean     std
model                                                 
Ensemble XGB+ResMLP  0.9650  0.0007     0.7951  0.0112
Logistic regression  0.7384  0.0010     0.0171  0.0082
Random forest        0.9550  0.0008     0.7263  0.0177
ResMLP               0.9597  0.0014     0.7700  0.0043
XGBoost              0.9615  0.0011     0.7737  0.0113
Ensemble - XGBoost PR-AUC per repeat: [0.0039, 0.0024, 0.0039, 0.0042, 0.0032] | mean 0.0035 | ensemble better in 5 of 5


... (8 outputs left out)

outer split 8 done


outer split 9 done



(B) Grouped 80/20 splits within development
                      PR-AUC                         Recall@P95                        
                       mean     std     min     max       mean     std     min     max
model                                                                                 
Ensemble XGB+ResMLP  0.9641  0.0068  0.9489  0.9734     0.7826  0.0315  0.7279  0.8254
Logistic regression  0.7352  0.0317  0.6862  0.7841     0.0318  0.0531  0.0023  0.1746
Random forest        0.9527  0.0083  0.9361  0.9644     0.7084  0.0426  0.6000  0.7409
ResMLP               0.9594  0.0074  0.9433  0.9683     0.7367  0.0776  0.5825  0.8367
XGBoost              0.9594  0.0080  0.9436  0.9707     0.7681  0.0365  0.6840  0.8023
Ensemble - XGBoost development holdout PR-AUC per split: [0.0053, 0.0095, 0.0024, 0.005, 0.0024, 0.0042, 0.0049, 0.0093, 0.0017, 0.0027] | mean 0.0047 | ensemble better in 10 of 10



(C) grouped vs ungrouped
             ungrouped 5-fold CV  grouped by star 5-fold CV
PR-AUC                   0.9625                     0.9607
Recall@P95               0.7834                     0.7691


## 8. Use the trained model
`python predict.py data/kepler_cumulative.csv` (see the README). The final models are in `models/final_xgboost.joblib` and `models/final_resmlp.joblib`.